In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense


In [3]:
df = pd.read_csv("finalData.csv")
df.head()

,Date,amount
0,2015-01-02,1526.0
1,2015-01-03,1260.0
2,2015-01-04,1694.0
3,2015-01-05,1530.0
4,2015-01-06,1495.0


In [5]:
# Step 1: Make Date a datetime column
df['Date'] = pd.to_datetime(df['Date'])

# Step 2: Sort by Date
df = df.sort_values('Date').reset_index(drop=True)


In [7]:
# Step 3: Extract the 'amount' column
amount_data = df['amount'].values.reshape(-1, 1)


In [9]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
amount_scaled = scaler.fit_transform(amount_data)


In [11]:
def create_sequences(data, n_steps):
    X, y = [], []
    for i in range(n_steps, len(data)):
        X.append(data[i-n_steps:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

n_steps = 30  # Using 30 past days
X, y = create_sequences(amount_scaled, n_steps)

# Reshape for LSTM input
X = X.reshape((X.shape[0], X.shape[1], 1))


In [13]:
df.head()

,Date,amount
0,2015-01-02,1526.0
1,2015-01-03,1260.0
2,2015-01-04,1694.0
3,2015-01-05,1530.0
4,2015-01-06,1495.0


In [15]:
model = Sequential()
model.add(LSTM(50, activation='relu', input_shape=(X.shape[1], X.shape[2])))
model.add(Dense(1))  # Output: Predicting next day's expense

model.compile(optimizer='adam', loss='mse')

# See the model structure
model.summary()

C:\Users\Ujjwal\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                          │ (None, 50)                  │          10,400 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 1)                   │              51 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 10,451 (40.82 KB)

 Trainable params: 10,451 (40.82 KB)

 Non-trainable params: 0 (0.00 B)

In [17]:
history = model.fit(X, y, epochs=50, batch_size=32, validation_split=0.2)


Epoch 1/50
91/91 ━━━━━━━━━━━━━━━━━━━━ 6s 24ms/step - loss: 0.0170 - val_loss: 0.0196
Epoch 2/50
91/91 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - loss: 0.0060 - val_loss: 0.0198
Epoch 3/50
91/91 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - loss: 0.0068 - val_loss: 0.0195
Epoch 4/50
91/91 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - loss: 0.0060 - val_loss: 0.0195
Epoch 5/50
91/91 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - loss: 0.0072 - val_loss: 0.0195
Epoch 6/50
91/91 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - loss: 0.0063 - val_loss: 0.0195
Epoch 7/50
91/91 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - loss: 0.0061 - val_loss: 0.0200
Epoch 8/50
91/91 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - loss: 0.0065 - val_loss: 0.0195
Epoch 9/50
91/91 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - loss: 0.0064 - val_loss: 0.0195
Epoch 10/50
91/91 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - loss: 0.0061 - val_loss: 0.0197
Epoch 11/50
91/91 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - loss: 0.0061 - val_loss: 0.0195
Epoch 12/50
91/91 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - loss: 0.0

In [19]:
# Predict the next day's expense

# Take the last 30 days from your scaled data
last_sequence = amount_scaled[-30:]
last_sequence = last_sequence.reshape((1, 30, 1))

# Predict
predicted_scaled = model.predict(last_sequence)
predicted_amount = scaler.inverse_transform(predicted_scaled)

print(f"Predicted next day's expense: ₹{predicted_amount[0][0]:.2f}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 275ms/step
Predicted next day's expense: ₹1100.83


In [21]:
model.save('lstm_expense_predictor.h5')


In [23]:
import pickle

# Save the scaler
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)


In [25]:
model.save('lstm_expense_predictor.keras')
